# Check a YAML report by hand

Runs `essay.yaml` through the loader and the docx back end, and shows what
comes out at every stage: the tasks, the finished document, and how it compares
with the reference captured from the old `.rdf`.

## Before you start

Pick the **`.venv313`** kernel of *this* worktree — `Scriptum-Report-dev-yaml`,
branch `dev-yaml`. Its editable install is what makes `import Scriptum` resolve
here rather than to `main`.

If that kernel is not offered, register it once:

```
E:/users/tel/Python/dev/Scriptum-Report-dev-yaml/.venv313/Scripts/python.exe -m pip install ipykernel
E:/users/tel/Python/dev/Scriptum-Report-dev-yaml/.venv313/Scripts/python.exe -m ipykernel install --user --name scriptum-dev-yaml --display-name "Scriptum dev-yaml (3.13)"
```

The next cell checks you are on the right one, so there is no need to guess.


In [1]:
import subprocess
import sys
from pathlib import Path

import Scriptum
import yaml

# Derived from where this notebook sits, so moving the worktree does not
# turn the check below into a false alarm. Jupyter starts in the notebook's
# own directory, and this cell runs before anything changes it.
NOTEBOOK_DIR = Path.cwd()
WORKTREE = NOTEBOOK_DIR.parents[2]
here = Path(Scriptum.__file__).resolve().parent.parent

print('python    ', sys.executable)
print('Scriptum  ', Path(Scriptum.__file__).resolve())
print('PyYAML    ', yaml.__version__)
print('branch    ', subprocess.run(['git', 'branch', '--show-current'], cwd=here,
                                   capture_output=True, text=True).stdout.strip())

if here != WORKTREE:
    print()
    print('WARNING: Scriptum is NOT coming from the dev-yaml worktree.')
    print('         Expected', WORKTREE)
    print('         Pick the .venv313 kernel of that worktree - see the cell above.')
else:
    print()
    print('OK: importing from the dev-yaml worktree.')


python     e:\users\tel\Python\dev\Scriptum-Report-dev-yaml\.venv313\Scripts\python.exe
Scriptum   E:\users\tel\Python\dev\Scriptum-Report-dev-yaml\Scriptum\__init__.py
PyYAML     6.0.3
branch     dev-yaml

OK: importing from the dev-yaml worktree.


## A workspace

The run writes a document and needs its data beside it, so everything is copied
to a temp directory. The repo stays clean, and you can throw the directory away.


In [2]:
import os
import shutil
import tempfile

REPORT_DIR = WORKTREE / 'tests' / '04_examples' / 'essay'

def workspace():
    """A fresh directory holding the fixtures, the template and the data."""
    work = Path(tempfile.mkdtemp(prefix='scriptum-'))
    for pattern in ('*.yaml', 'essay.docx'):
        for path in REPORT_DIR.glob(pattern):
            shutil.copy(path, work)
    shutil.copytree(REPORT_DIR / 'data', work / 'data', dirs_exist_ok=True)
    os.chdir(work)
    return work

WORK = workspace()
print('working in', WORK)
print(sorted(p.name for p in WORK.iterdir()))


working in C:\Users\tel\AppData\Local\Temp\scriptum-c0sn4deg
['data', 'essay.docx', 'essay.yaml']


## 1. Read the document

`ReportDataFile` reads a `.yaml` document through `Scriptum.rdf.loader`;
anything else is refused with a message. The `.rdf` text parser is gone.

A broken document raises `DocumentError` carrying **every** diagnostic, not the
first — there is a cell near the bottom that shows one.


In [3]:
rdf = Scriptum.ReportDataFile('essay.yaml')

print('documenttype:', rdf.settings.documenttype)
print('datadir     :', rdf.settings.datadir)
print('tasks       :', len(rdf.tasks))
print('errors      :', rdf.errors or 'none')


documenttype: docx
datadir     : data
tasks       : 56
errors      : none


## 2. What the tasks say

`what` is the operation, `where` the marker an *add* lands at, `target` the
**template name** — the tag written in the .docx — and the address the
**instance**: `subsection:instruction::2` is the second copy of that block.

Note the `_global_` tasks at the end: global fills are applied last, and the
task list carries that rule so no back end has to remember it.


In [4]:
def show_tasks(tasks, limit=None, only=None):
    rows = [t for t in tasks if only is None or only in '.'.join(t.myAddress)]
    print(f'{"#":>4}  {"what":6} {"where":18} {"target":22} address')
    print('-' * 110)
    for t in rows[:limit]:
        print(f'{t.serial:>4}  {t.what or "-":6} {t.where or "-":18} '
              f'{t.target or "-":22} {".".join(t.myAddress)}')
    if limit and len(rows) > limit:
        print(f'... {len(rows) - limit} more')

show_tasks(rdf.tasks, limit=40)


   #  what   where              target                 address
--------------------------------------------------------------------------------------------------------------
   1  apply  -                  -                      section:main::1
   2  -      -                  title                  section:main::1.:title::1
   3  -      -                  subtitle               section:main::1.:subtitle::1
   4  -      -                  authors                section:main::1.:authors::1
   5  apply  -                  -                      section:main::1.subsection:intro::1
   6  -      -                  text                   section:main::1.subsection:intro::1.:text::1
   7  apply  -                  -                      section:main::1.subsection:content::1
   8  -      -                  head                   section:main::1.subsection:content::1.:head::1
   9  -      -                  text                   section:main::1.subsection:content::1.:text::1
  10  add    marker

Try `show_tasks(rdf.tasks, only='subsection:conent')` to see just the
repeated block — instance 1 **applies** to what the template already holds, and
2 and 3 and so on are **copies**.


In [5]:
show_tasks(rdf.tasks, only='subsection:content')


   #  what   where              target                 address
--------------------------------------------------------------------------------------------------------------
   7  apply  -                  -                      section:main::1.subsection:content::1
   8  -      -                  head                   section:main::1.subsection:content::1.:head::1
   9  -      -                  text                   section:main::1.subsection:content::1.:text::1
  10  add    marker:content::1  image:generic          section:main::1.subsection:content::1.image:generic::1
  11  add    marker:content::1  image:generic          section:main::1.subsection:content::1.image:generic::2
  12  add    marker:content::1  text:generic           section:main::1.subsection:content::1.text:generic::1
  13  copy   -                  -                      section:main::1.subsection:content::2
  14  -      -                  head                   section:main::1.subsection:content::2.:head::1
  15 

## 3. Build the document

Anything the back end could not place prints a `WARNING`. A clean run prints
none.


In [6]:
import contextlib
import io as _io

with contextlib.redirect_stdout(_io.StringIO()) as printed:
    managed = Scriptum.ManagedDocx('essay.docx', rdf)
    managed.typesetting(rdf)
    managed.save('final_essay.docx',finish=True)

complaints = [line for line in printed.getvalue().splitlines()
              if 'WARNING' in line or 'ERROR' in line]
print('written:', WORK / 'final_essay.docx')
print('complaints:', len(complaints))
for line in complaints[:20]:
    print('  ', line)


written: C:\Users\tel\AppData\Local\Temp\scriptum-c0sn4deg\final_essay.docx
complaints: 0


## 4. Read it back

Open `final_essay.docx` in Word if you want to look at the formatting; this shows
what it *says*, which is what the automated comparison uses.


In [7]:
import docx

def spoken(path):
    document = docx.Document(path)
    said = [p.text.strip() for p in document.paragraphs]
    for table in document.tables:
        for row in table.rows:
            said.extend(cell.text.strip() for cell in row.cells)
    return [line for line in said if line]

lines = spoken(WORK / 'final_essay.docx')
print(len(lines), 'non-empty lines')
for line in lines[:30]:
    print('  ', line[:100])


82 non-empty lines
   From Typewriting to Variable Fonts:
   History, Technology, and Contemporary Letter Design
   Authors: Scriptum, ChatGPT, Google, Wikipedia
   Introduction
   Typewriting is both a mechanical craft and a cultural practice. For more than a century, the rhythmi
   Early Experiments and the Birth of the Typewriter
   Attempts to mechanize writing predate the modern typewriter by centuries. Inventors in eighteenth- a
   Figure 1: The Sholes and Glidden (1876) - later Remington #1
   Figure 2: Remington #2 with a shift key
   It introduced key elements that would define the industry: a cylindrical platen to hold the paper, a
   Standardization, Visible Writing, and Social Effects
   In the late nineteenth and early twentieth centuries, competing manufacturers refined typewriter des
   Standardization in typewriter design reshaped business communication. Typewritten documents offered 
   The Rise of Electric Typewriters
   By the mid-twentieth century, manufacturers sou

## 5. Compare with the reference

`expected/essay.json`, beside this notebook, is what this
fixture's **`.rdf`** produced before the back end changed (the `.rdf` and its
parser are gone; the reference is their record). Digits and weekday
names are collapsed on both sides, because the reference was captured on
another day and `date:now` is evaluated per run.

An empty report below means the YAML document says exactly what the text one
said.


In [8]:
import json
import re

DIGITS = re.compile(r'\d+')
WEEKDAY = re.compile(r'\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\b')

def normalise(lines):
    return [WEEKDAY.sub('#', DIGITS.sub('#', line)) for line in lines]

REFERENCE = REPORT_DIR / 'expected' / 'essay.json'

expected = normalise(json.loads(REFERENCE.read_text(encoding='utf-8')))
got = normalise(spoken(WORK / 'final_essay.docx'))

print(f'reference {len(expected)} lines, this run {len(got)} lines')
if expected == got:
    print('IDENTICAL')
else:
    for i, (a, b) in enumerate(zip(expected, got)):
        if a != b:
            print(f'first difference at line {i}')
            print('  reference:', a[:110])
            print('  this run :', b[:110])
            break
    else:
        print('one is a prefix of the other')


reference 82 lines, this run 82 lines
IDENTICAL
